# Chirp v3 Batch Transcription Pipeline

This Colab orchestrates the large-scale transcription of audio segments using Google Cloud's **Chirp v3** (Speech-to-Text v2) model.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/chirp_transcribe_audio.ipynb)

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically filters out files without speech to optimize API costs.
2.  **Real-time Monitoring**: Provides a lightweight polling dashboard that shows batch progress, server update times, and a bulleted list of truly outstanding files by cross-referencing existing result blobs in GCS.
3.  **Error Diagnostics**: Includes a diagnostic tool to scan completed operations for file-level rejections or API errors.

In [ ]:
# @title Install dependencies
%pip install loguru

In [ ]:
# @title Import dependencies
import json
import sys
import time
from pathlib import Path

import google.api_core.exceptions
from google.api_core import client_options
from google.cloud import storage
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech
from google.colab import auth
from google.longrunning import operations_pb2
from IPython.display import clear_output
from loguru import logger

In [ ]:
# @title Input and Constants
MODEL_ID = "chirp_3"  # @param ["chirp_3", "chirp_telephony"] {type:"string"}
GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
PROJECT_NAME = ""  # @param {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}

assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided and cannot be empty."
assert GCS_BUCKET, "GCS_BUCKET must be provided and cannot be empty."
assert PROJECT_NAME, "PROJECT_NAME must be provided and cannot be empty."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."

GCS_INPUT_DIR = f"segmented_audio/{PROJECT_NAME}_audio"
# @markdown Enable if modifications (e.g. silence padding) were applied during segmentation:
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
AUDIO_DENOISING = True  # @param {type:"boolean"}
OVERWRITE_EXISTING_OUTPUT = True  # @param {type:"boolean"}
ENABLE_PHRASE_SETS = False  # @param {type:"boolean"}
# @markdown **Note:** Word time offsets will cause the pipeline to fail if phrase sets are used.
ENABLE_WORD_TIME_OFFSETS = True  # @param {type:"boolean"}
# @markdown **Note:** Use `0` below to indicate unlimited/all batches.
MAX_BATCHES_TO_RUN = 0  # @param {type:"integer"}

if MAX_BATCHES_TO_RUN <= 0:
    MAX_BATCHES_TO_RUN = None

if not AUDIO_PREPROCESSING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"

LOCATION = "us" if MODEL_ID == "chirp_3" else "us-central1"

GCS_OUTPUT_DIR = (
    f"transcripts/{PROJECT_NAME}_audio/{MODEL_ID}/{EXPERIMENT_NAME}"
)

assert ENABLE_PHRASE_SETS != ENABLE_WORD_TIME_OFFSETS, (
    "Word time offsets and phrase sets cannot both be enabled."
)

MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"

# Prompt based on Chirp "default prompt" with fire-radio context
# NOTE: Keep BASE_PHRASE_SET in sync with:
# backend/pipeline/transcription/chirp_phrase_hints.txt
_BASE_CUSTOM_PROMPT = """\
Evaluate all audio specifically as VHF/UHF fire-related dispatch radio traffic. The speakers will use heavy jargon, but you must transcribe EVERY spoken word, including conversational phrasing and incomplete sentences.
This audio likely contains mic clicks, RF static, radio hum, and possibly some unintelligible speech. Only transcribe intelligible speech.

CRITICAL RULES:
* If the audio is completely unintelligible, output the following: [UNINTELLIGIBLE]
* Output the transcript exactly as said, with no newlines.
* Do not continue the speech segment beyond what is spoken.
* When transcribing numbers, write the digits grouped together (e.g., 100 instead of one hundred, 6333 instead of 63 33).
* Format all unit identifiers as the unit type followed by digits (e.g., Engine 41, Battalion 2, Medic 12).
"""

if ENABLE_WORD_TIME_OFFSETS:
    # Chirp appends a word-time-offsets instruction to the end of the custom
    # prompt, so add an asterisk to incorporate it into the critical rules
    # list: "Include the end timestamp for each word in ctrl token format
    # using numerals from <ctrl1> to <ctrl64>."
    CUSTOM_PROMPT = _BASE_CUSTOM_PROMPT + "* "
else:
    CUSTOM_PROMPT = _BASE_CUSTOM_PROMPT

# NOTE: Keep BASE_PHRASE_SET in sync with:
# backend/pipeline/transcription/chirp_phrase_hints.txt
BASE_PHRASE_SET = [
    # Status & Acknowledgments
    "copy",
    "received",
    "affirmative",
    "affirm",
    "proceed",
    "responding",
    "responding to",
    "en-route",
    "on-scene",
    "on-scene in the area",
    "available",
    "returning",
    "in service",
    "got a caller",
    "caller advising",
    "in quarters",
    "arrived",
    "go ahead",
    "back at",
    # Apparatus & Unit Designators
    "engine",
    "tanker",
    "brush",
    "brush truck",
    "tender",
    "battalion",
    "squad",
    "ladder",
    "tower",
    "tower-ladder",
    "medic",
    "ambulance",
    "k",
    "branch",
    "chopper",
    "copter",
    # Tactical Jargon & Acronyms
    "AIQ",
    "AOR",
    "IC",
    "ICP",
    "LAT",
    "RP",
    "SEAT",
    "TAC",
    "VFIRE",
    "VLAT",
    "air attack",
    "air tactics",
    "helispot",
    "lead plane",
    "strike team",
    "control",
    # Fire Behavior & Benchmarks
    "being toned",
    "box alarm",
    "cancel the balance",
    "chaparral",
    "exposure protection",
    "fire attack",
    "fire boss",
    "forward progress stopped",
    "forward rate of spread stopped",
    "heavy timber",
    "left flank",
    "light flashy fuels",
    "rate of spread",
    "right flank",
    "structure defense",
    "structure protection",
    "structures threatened",
    "terrain driven",
    "wind driven",
]

# APCO 10-Codes
TEN_CODES = ["10-4", "10-7", "10-8", "10-9", "10-20", "10-22", "10-23", "10-97"]

# Final flat list for the API
FINAL_PHRASE_LIST = BASE_PHRASE_SET + TEN_CODES

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [ ]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Grant Chirp Speech API access to the GCS bucket
# Run once per project to allow the Speech-to-Text service agent to read audio from GCS.
project_number = !gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)'
speech_sa = f"service-{project_number[0]}@gcp-sa-speech.iam.gserviceaccount.com"
!gsutil iam ch serviceAccount:{speech_sa}:objectAdmin gs://{GCS_BUCKET}

In [ ]:
# @title Helper functions
def start_chirp_jobs_from_manifest(
    manifest_gcs_uri: str,
    project_id: str | None = None,
    *,
    overwrite: bool = False,
    max_batches: int | None = None,
) -> tuple[list[str], int, dict[str, list[str]]]:
    """
    Submits transcription jobs using explicit phrase expansion and custom prompts.
    """
    project_id = project_id or GCP_PROJECT_ID
    endpoint = f"{LOCATION}-speech.googleapis.com"
    opts = client_options.ClientOptions(
        api_endpoint=endpoint, quota_project_id=project_id
    )
    SpeechClient(client_options=opts)
    storage_client = storage.Client(project=project_id)
    batch_timestamp = int(time.time())

    logger.info(f"Reading manifest from GCS: {manifest_gcs_uri}")
    bucket_name = manifest_gcs_uri.replace("gs://", "").split("/")[0]
    blob_path = "/".join(manifest_gcs_uri.replace("gs://", "").split("/")[1:])
    bucket = storage_client.bucket(bucket_name)
    content = bucket.blob(blob_path).download_as_text()
    manifest_entries = [
        json.loads(line) for line in content.strip().split("\n") if line.strip()
    ]

    # Handle Overwrite Logic
    if overwrite:
        logger.warning(
            f"Overwrite enabled. Deleting existing blobs in {GCS_OUTPUT_DIR}..."
        )
        blobs_to_delete = list(
            storage_client.list_blobs(GCS_BUCKET, prefix=f"{GCS_OUTPUT_DIR}/")
        )
        if blobs_to_delete:
            bucket.delete_blobs(blobs_to_delete)
            logger.info(f"Deleted {len(blobs_to_delete)} existing blobs.")

    existing_blobs = list(
        storage_client.list_blobs(GCS_BUCKET, prefix=f"{GCS_OUTPUT_DIR}/")
    )
    existing_basenames = {
        Path(b.name).name for b in existing_blobs if b.name.endswith(".json")
    }
    pending_audio_uris = []
    for entry in manifest_entries:
        uri = entry["audio_filepath"]
        if not overwrite and any(
            Path(uri).name.replace(".flac", "") in res
            for res in existing_basenames
        ):
            continue
        pending_audio_uris.append(uri)

    if not pending_audio_uris:
        logger.info("No pending files to process.")
        return [], batch_timestamp, {}

    return submit_specific_uris(
        pending_audio_uris, project_id, batch_timestamp, max_batches=max_batches
    )


def submit_specific_uris(
    uri_list: list[str],
    project_id: str,
    batch_timestamp: int,
    max_batches: int | None = None,
) -> tuple[list[str], int, dict[str, list[str]]]:
    """Core submission logic for a list of GCS audio URIs."""
    client = SpeechClient(
        client_options=client_options.ClientOptions(
            api_endpoint=f"{LOCATION}-speech.googleapis.com",
            quota_project_id=project_id,
        )
    )

    phrases = [
        cloud_speech.PhraseSet.Phrase(value=p) for p in FINAL_PHRASE_LIST
    ]
    adaptation = cloud_speech.SpeechAdaptation(
        phrase_sets=[
            cloud_speech.SpeechAdaptation.AdaptationPhraseSet(
                inline_phrase_set=cloud_speech.PhraseSet(phrases=phrases)
            )
        ]
    )

    recognizer_path = (
        f"projects/{project_id}/locations/{LOCATION}/recognizers/_"
    )
    batch_size = 15
    chunks = [
        uri_list[i : i + batch_size]
        for i in range(0, len(uri_list), batch_size)
    ][:max_batches]
    operation_names, op_to_filenames = [], {}

    for i, segment_batch in enumerate(chunks):
        request = cloud_speech.BatchRecognizeRequest(
            recognizer=recognizer_path,
            config=cloud_speech.RecognitionConfig(
                auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
                model=MODEL_ID,
                language_codes=["en-US"],
                adaptation=adaptation,
                features=cloud_speech.RecognitionFeatures(
                    enable_automatic_punctuation=True,
                    enable_word_time_offsets=ENABLE_WORD_TIME_OFFSETS,
                    custom_prompt_config=cloud_speech.CustomPromptConfig(
                        custom_prompt=CUSTOM_PROMPT
                    ),
                ),
                denoiser_config=cloud_speech.DenoiserConfig(
                    denoise_audio=AUDIO_DENOISING
                ),
            ),
            files=[
                cloud_speech.BatchRecognizeFileMetadata(uri=u)
                for u in segment_batch
            ],
            recognition_output_config=cloud_speech.RecognitionOutputConfig(
                gcs_output_config=cloud_speech.GcsOutputConfig(
                    uri=f"gs://{GCS_BUCKET}/{GCS_OUTPUT_DIR}"
                )
            ),
        )
        try:
            op = client.batch_recognize(request=request)
            operation_names.append(op._operation.name)
            op_to_filenames[op._operation.name] = segment_batch
            logger.info(f"Submitted Batch {i} ({len(segment_batch)} files)")
        except Exception as e:
            logger.error(f"Batch submission failed: {e}")
    return operation_names, batch_timestamp, op_to_filenames


def run_chirp_retry_pipeline(
    operation_names: list[str],
) -> tuple[list[str], int, dict[str, list[str]]]:
    """Orchestrates identification of failures and re-submission."""
    client = SpeechClient(
        client_options=client_options.ClientOptions(
            api_endpoint=f"{LOCATION}-speech.googleapis.com",
            quota_project_id=GCP_PROJECT_ID,
        )
    )
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket = storage_client.bucket(GCS_BUCKET)
    from google.longrunning import operations_pb2

    confirmed_failed_uris = []
    for op_name in operation_names:
        op = client.get_operation(
            request=operations_pb2.GetOperationRequest(name=op_name)
        )
        if op.done and op.response:
            resp = cloud_speech.BatchRecognizeResponse.deserialize(
                op.response.value
            )
            for uri, result in resp.results.items():
                if result.error and result.error.code != 0:
                    filename = Path(uri).name.replace(".flac", "")
                    prefix = f"{GCS_OUTPUT_DIR}/{filename}_transcript"
                    if not list(
                        bucket.list_blobs(prefix=prefix, max_results=1)
                    ):
                        confirmed_failed_uris.append(uri)

    if confirmed_failed_uris:
        logger.info(
            f"Starting retry for {len(confirmed_failed_uris)} confirmed failures..."
        )
        return submit_specific_uris(
            confirmed_failed_uris, GCP_PROJECT_ID, int(time.time())
        )
    logger.info("No confirmed failures to retry.")
    return [], 0, {}


def poll_progress() -> None:
    """
    Polls the status of all submitted operations.
    Includes exponential backoff for quota limits.
    """
    try:
        if "operations" in globals() and operations:
            client = SpeechClient(
                client_options=client_options.ClientOptions(
                    api_endpoint=f"{LOCATION}-speech.googleapis.com",
                    quota_project_id=GCP_PROJECT_ID,
                )
            )

            logger.info(f"--- Chirp Status | Batch ID: {timestamp} ---")

            while True:
                completed_count = 0
                for op_name in operations:
                    op = client.get_operation(
                        request=operations_pb2.GetOperationRequest(name=op_name)
                    )
                    if op.done:
                        completed_count += 1

                clear_output(wait=True)
                logger.info(
                    f"Progress: {completed_count}/{len(operations)} batches complete."
                )

                if completed_count == len(operations):
                    logger.info("All batches finished!")
                    break

                time.sleep(30)
        else:
            logger.info("No active operations found to poll.")
    except google.api_core.exceptions.ResourceExhausted:
        logger.warning("Quota exceeded. Sleeping for 60s...")
        time.sleep(60)
        poll_progress()
    except Exception as e:
        logger.error(f"Polling error: {e}")


def audit_with_error_analysis() -> None:
    if "operations" not in globals() or not operations:
        logger.info(
            "No active operations to audit. Run the batch submission cell first."
        )
        return
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket = storage_client.bucket(GCS_BUCKET)
    endpoint = f"{LOCATION}-speech.googleapis.com"
    opts = client_options.ClientOptions(
        api_endpoint=endpoint, quota_project_id=GCP_PROJECT_ID
    )
    client = SpeechClient(client_options=opts)
    from google.longrunning import operations_pb2

    # 1. Manifest data
    manifest_blob = bucket.blob(MANIFEST_URI.replace(f"gs://{GCS_BUCKET}/", ""))
    manifest_content = manifest_blob.download_as_text()
    intended_files = [
        json.loads(line)["audio_filepath"]
        for line in manifest_content.strip().split("\n")
        if line.strip()
    ]

    # 2. Inspect Operations for Errors
    failed_uris = []
    internal_error_count = 0

    logger.info(
        f"Auditing {len(operations)} operations for internal file-level errors..."
    )
    for op_name in operations:
        try:
            op = client.get_operation(
                request=operations_pb2.GetOperationRequest(name=op_name)
            )
            if op.done and op.response:
                resp = cloud_speech.BatchRecognizeResponse.deserialize(
                    op.response.value
                )
                for uri, result in resp.results.items():
                    if result.error.code != 0:
                        internal_error_count += 1
                        failed_uris.append(uri)
            time.sleep(0.6)
        except Exception as e:
            logger.error(f"Could not check op {op_name}: {e}")

    # 3. Check GCS physical results
    existing_blobs = list(
        storage_client.list_blobs(GCS_BUCKET, prefix=GCS_OUTPUT_DIR)
    )
    blob_names = [
        Path(b.name).name for b in existing_blobs if b.name.endswith(".json")
    ]

    logger.info(f"\n--- Audit Report: {EXPERIMENT_NAME} ---")
    logger.info(f"Total Files in Manifest: {len(intended_files)}")
    logger.info(f"Files with API Internal Errors: {internal_error_count}")
    logger.info(f"JSON Files found in GCS: {len(blob_names)}")

    if internal_error_count > 0 or len(blob_names) < len(intended_files):
        logger.error(
            f"\n[DIAGNOSIS]: Found {internal_error_count} file-level failures."
        )
        logger.error(
            "[ACTION]: Run the 'Run Automated Retry Pipeline' cell to re-process these files."
        )

In [ ]:
# @title Execute batch jobs
operations, timestamp, op_to_filenames = start_chirp_jobs_from_manifest(
    MANIFEST_URI,
    overwrite=OVERWRITE_EXISTING_OUTPUT,
    max_batches=MAX_BATCHES_TO_RUN,
)

logger.info("\n--- Batch Submission Summary ---")
logger.info(f"Batch ID: {timestamp}")
logger.info(f"Total Operations started: {len(operations)}")
logger.info(f"Results: gs://{GCS_BUCKET}/{GCS_OUTPUT_DIR}/")

In [ ]:
# @title Polls the progress of the pipeline
poll_progress()

In [ ]:
# @title Run Automated Retry Pipeline
# Identifies failures and re-submits them automatically
new_ops, _, op_to_filenames_retry = run_chirp_retry_pipeline(operations)

if new_ops:
    # We append to the existing timestamp to keep the session linked
    timestamp = f"{timestamp}_retry"
    logger.info(f"Retrying {len(new_ops)} batches. Session ID: {timestamp}")

    # Update global state so poll_progress sees the new batch
    operations = new_ops
    poll_progress()
else:
    logger.info("No further retries needed. All files accounted for.")

In [ ]:
# @title Final Pipeline Health Check
# This cell performs a final audit. It reads the original manifest and checks
# GCS to see exactly how many files are missing, regardless of what the API
# reported.

audit_with_error_analysis()

In [ ]:
# @title Write run metadata
import datetime as _datetime

_run_metadata = {
    "custom_prompt": CUSTOM_PROMPT,
    "base_phrase_set": BASE_PHRASE_SET,
    "ten_codes": TEN_CODES,
    "model_id": MODEL_ID,
    "location": LOCATION,
    "audio_preprocessing": AUDIO_PREPROCESSING,
    "audio_denoising": AUDIO_DENOISING,
    "enable_word_time_offsets": ENABLE_WORD_TIME_OFFSETS,
    "enable_phrase_sets": ENABLE_PHRASE_SETS,
    "project_name": PROJECT_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "input_manifest_uri": MANIFEST_URI,
    "output_dir": f"gs://{GCS_BUCKET}/{GCS_OUTPUT_DIR}",
    "run_timestamp_utc": _datetime.datetime.now(
        _datetime.timezone.utc
    ).strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
}

_meta_path = f"{GCS_OUTPUT_DIR}/run_metadata.json"
storage.Client(project=GCP_PROJECT_ID).bucket(GCS_BUCKET).blob(
    _meta_path
).upload_from_string(
    json.dumps(_run_metadata, indent=2),
    content_type="application/json",
)
logger.info(f"Run metadata written to gs://{GCS_BUCKET}/{_meta_path}")